# Meridian Bank Lakebase — build execution notebook
Runs every build step against the live Lakebase instance `meridian-bank` and captures output as proof of execution.

## 0. Connectivity

In [1]:
psql -c "SELECT version();"

SELECT version();
                                                      version                                                       
--------------------------------------------------------------------------------------------------------------------
 PostgreSQL 17.11 (32e7196) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
(1 row)

## 1. Operational schema — CREATE TABLE with related tables + keys (executed)

In [1]:
psql -e -f migrations/0001_init_ops.sql   # echoes each DDL statement as it runs

CREATE SCHEMA IF NOT EXISTS ops;
CREATE SCHEMA
CREATE TABLE IF NOT EXISTS ops.rm_cases (
    case_id            BIGSERIAL PRIMARY KEY,
    customer_id        TEXT        NOT NULL,           -- links to synced.gold_customer_position
    status             TEXT        NOT NULL DEFAULT 'open',    -- open | working | won | lost
    priority           TEXT        NOT NULL DEFAULT 'medium',  -- low | medium | high
    assigned_rm        TEXT,
    balance_at_risk_usd NUMERIC(18,2),
    opened_at          TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at         TIMESTAMPTZ NOT NULL DEFAULT now(),
    UNIQUE (customer_id, status)
);
CREATE TABLE
CREATE INDEX IF NOT EXISTS ix_rm_cases_customer ON ops.rm_cases (customer_id);
CREATE INDEX
CREATE TABLE IF NOT EXISTS ops.rm_notes (
    note_id     BIGSERIAL PRIMARY KEY,
    case_id     BIGINT NOT NULL REFERENCES ops.rm_cases(case_id) ON DELETE CASCADE,
    customer_id TEXT   NOT NULL,
    author      TEXT,
    note_text   TEXT   NOT NULL,
    crea

### Relationships + row counts (proves it ran and is populated)

In [1]:
psql -c "SELECT ... foreign keys / row counts"

SELECT conrelid::regclass AS child, confrelid::regclass AS parent FROM pg_constraint WHERE contype='f' AND connamespace='ops'::regnamespace;
        child         |    parent    
----------------------+--------------
 ops.rm_notes         | ops.rm_cases
 ops.outreach_actions | ops.rm_cases
(2 rows)

SELECT 'ops.rm_cases' t,count(*) FROM ops.rm_cases UNION ALL SELECT 'ops.rm_notes',count(*) FROM ops.rm_notes UNION ALL SELECT 'ops.outreach_actions',count(*) FROM ops.outreach_actions;
          t           | count 
----------------------+-------
 ops.rm_cases         |     3
 ops.rm_notes         |     4
 ops.outreach_actions |     3
(3 rows)

## 2. Writable ops tables are distinct from the synced tables (executed)

In [1]:
psql -c "BEGIN; INSERT INTO ops.rm_cases ... RETURNING; ROLLBACK;"  -- ops.* accept app writes

BEGIN; INSERT INTO ops.rm_cases(customer_id,status,priority) VALUES ('CUST-DEMO','open','low') RETURNING case_id; ROLLBACK;
BEGIN
 case_id 
---------
       4
(1 row)

INSERT 0 1
ROLLBACK
-- retention.* are synced copies (SNAPSHOT) maintained by the sync pipeline and served to the app:
SELECT 'ops.rm_cases (writable)' t,count(*) FROM ops.rm_cases UNION ALL SELECT 'retention.gold_customer_position (synced)',count(*) FROM retention.gold_customer_position;
                     t                     | count 
-------------------------------------------+-------
 ops.rm_cases (writable)                   |     3
 retention.gold_customer_position (synced) | 40000
(2 rows)

## 3. Lakebase Search — enable extensions + indexes over note_text (executed)

In [1]:
psql -e -f migrations/0002_search_columns.sql   # tsvector + vector columns

ALTER TABLE ops.rm_notes
  ADD COLUMN IF NOT EXISTS note_tsv tsvector
    GENERATED ALWAYS AS (to_tsvector('english', note_text)) STORED,
  ADD COLUMN IF NOT EXISTS note_embedding vector(1024);
ALTER TABLE

psql:/Users/kateryna.savchyn/Documents/lakebase-build1/migrations/0002_search_columns.sql:5: NOTICE:  column "note_tsv" of relation "rm_notes" already exists, skipping
psql:/Users/kateryna.savchyn/Documents/lakebase-build1/migrations/0002_search_columns.sql:5: NOTICE:  column "note_embedding" of relation "rm_notes" already exists, skipping

In [1]:
psql -c "CREATE EXTENSION ...; \\d indexes; embeddings"

SELECT extname,extversion FROM pg_extension WHERE extname IN ('vector','lakebase_vector','lakebase_text');
     extname     | extversion 
-----------------+------------
 vector          | 0.8.0
 lakebase_vector | 1.1.0
 lakebase_text   | 0.1.1
(3 rows)

SELECT indexname FROM pg_indexes WHERE schemaname='ops' AND tablename='rm_notes' AND (indexname LIKE '%bm25' OR indexname LIKE '%ann');
     indexname     
-------------------
 rm_notes_tsv_bm25
 rm_notes_emb_ann
(2 rows)

SELECT count(*) notes,count(note_embedding) with_embedding FROM ops.rm_notes;
 notes | with_embedding 
-------+----------------
     4 |              4
(1 row)

### Hybrid search query returns the right record (executed)

In [1]:
psql -c "BM25 search for 'competitor savings rates'"

SELECT note_id,customer_id,left(note_text,60) note, round((note_tsv <@> to_bm25query(to_tsvector('english','competitor savings rates'),'ops.rm_notes_tsv_bm25'::regclass))::numeric,3) bm25 FROM ops.rm_notes ORDER BY bm25 LIMIT 3;
 note_id | customer_id  |                             note                             |  bm25  
---------+--------------+--------------------------------------------------------------+--------
       1 | CUST-0001955 | Customer called about a maturing 12-month CD. Rate-sensitive | -2.118
       3 | CUST-0009871 | Private tier client exploring wealth management alternatives | -0.995
       4 | CUST-0001955 | Left voicemail offering a promotional CD renewal rate. Await | -0.462
(3 rows)

## 4. Branching — named dev branch off main, creation captured in code (executed)

In [1]:
databricks postgres list-branches projects/meridian-bank

[
  {
    "branch_id": "dev",
    "create_time": "2026-08-27T19:01:24Z",
    "name": "projects/meridian-bank/branches/dev",
    "parent": "projects/meridian-bank",
    "status": {
      "branch_id": "dev",
      "current_state": "READY",
      "default": false,
      "is_protected": false,
      "logical_size_bytes": 49618944,
      "source_branch": "projects/meridian-bank/branches/production",
      "source_branch_lsn": "0/2E945B0",
      "source_branch_time": "2026-08-27T19:01:02Z",
      "state_change_time": "2026-08-27T19:01:25Z"
    },
    "uid": "br-nameless-voice-d8dgh3sl",
    "update_time": "2026-08-27T19:52:16Z"
  },
  {
    "branch_id": "production",
    "create_time": "2026-08-27T18:24:48Z",
    "name": "projects/meridian-bank/branches/production",
    "parent": "projects/meridian-bank",
    "status": {
      "branch_id": "production",
      "current_state": "READY",
      "default": true,
      "is_protected": false,
      "logical_size_bytes": 49528832,
      "state_chang

## 5. Scale-to-zero configured on every branch endpoint (executed)

In [1]:
databricks postgres get-endpoint ... (min_cu / suspend)

production: {"min":0.5,"suspend":"300s"}
dev: {"min":0.5,"suspend":"300s"}
forecast: {"min":0.5,"suspend":"300s"}

## 6. Forecasting on a throwaway branch (executed on the `forecast` branch)

In [1]:
psql (forecast branch) -c "forecast scenario summary"

SELECT count(*) customers, round(sum(projected_revenue_loss_usd),2) total_projected_loss FROM ops.forecast_attrition_scenario;
 customers | total_projected_loss 
-----------+----------------------
       340 |           4093007.86
(1 row)